<a href="https://colab.research.google.com/github/sumaiyanawaz776-code/Machine-Learning-Advance/blob/main/News_Classification_with_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers datasets evaluate accelerate gradio

In [ ]:
import torch

In [ ]:
print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)

import gradio as gr
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
dataset = load_dataset("ag_news")
print(dataset)
print(dataset["train"][0])

In [ ]:
train_data = dataset["train"].shuffle(seed=42).select(range(8000))
test_data = dataset["test"].shuffle(seed=42).select(range(2000))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(batch):
  return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_data = train_data.map(tokenize_function, batched=True)
test_data = test_data.map(tokenize_function, batched=True)


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4,
    id2label={0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"},
    label2id={"World": 0, "Sports": 1, "Business": 2, "Sci/Tech": 3}
)

In [ ]:
!pip install -q evaluate

In [ ]:
import evaluate

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=True,
    dataloader_num_workers=2
)


accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)
  acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
  f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
  return {"accuracy": acc, "f1_score": f1}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics,
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(results)

In [ ]:
predictions = trainer.predict(test_data)
preds = np.argmax(predictions.predictions, axis=-1)
print("Test Accuracy:", accuracy_metric.compute(predictions=preds, references=test_data["label"])["accuracy"])
print("Test F1 Score:", f1_metric.compute(predictions=preds, references=test_data["label"], average="weighted")["f1"])

In [ ]:
trainer.save_model("./result")
tokenizer.save_pretrained("./result")

In [ ]:
classifier = pipeline(
    "text-classification",
    model="./bert-ag-news",
    tokenizer="./bert-ag-news",
    device=0 if torch.cuda.is_available() else -1
)

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("./bert-ag-news")
id2label = config.id2label

def predict_news(text):
  result = classifier(text, truncation=True, max_length=128)[0]
  label = result["label"]
  return f"Predicted Label:  {label}"

In [ ]:
demo = gr.Interface(
    fn=predict_news,
    inputs=gr.Textbox(lines=5, placeholder="Enter news text here..."),
    outputs=gr.Markdown(),
    title="News Classification with BERT",
    description="Enter a news article and get its classification",
    examples=[
        ["Apple unveils new AI-powered MacBook with M4 chip"],
        ["Real Madrid defeats Barcelona in El Clasico thriller"],
        ["Fed raises interest rates amid inflation concerns"],
        ["NASA confirms discovery of potential signs of life on Mars"],
        ["World leaders gather for climate summit in Geneva"]
    ],
    allow_flagging="never",
    theme="soft"
)

demo.launch(
    share=True,
    debug=True,
    show_error=True
    )